# 03 · Gold
Business-ready tables:
- `customer_monthly`: single customer view per month
- `kpi_customers_monthly`, `kpi_sales_monthly`: dashboard KPIs
- `propensity_features`: model input for `pension_plan` propensity

In [0]:
from pyspark.sql import functions as F

from easymoney_pipeline.quality import (
    DataQualityError, check_in_domain, check_row_count, check_unique,
)
from easymoney_pipeline.transforms import add_customer_timeline, align_sales_to_snapshot

SILVER = "workspace.easymoney_silver"
GOLD = "workspace.easymoney_gold"
KEYS = ["customer_id", "partition_month"]

TARGET = "pension_plan"                                # 8% of sales, 79% of net margin
EXCLUDED_PRODUCTS = ["em_account_p", "em_account_pp"]  # 0% penetration


def write_gold(df, table: str, comment: str) -> int:
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", True)
       .saveAsTable(f"{GOLD}.{table}"))
    spark.sql(f"COMMENT ON TABLE {GOLD}.{table} IS '{comment}'")
    n = spark.table(f"{GOLD}.{table}").count()
    print(f"{table:<30} {n:>12,} rows")
    return n

## customer_monthly

In [0]:
products = spark.table(f"{SILVER}.customer_products")
PRODUCT_COLS = [c for c in products.columns if c not in KEYS]

customer_monthly = (
    products
    .join(spark.table(f"{SILVER}.customer_sociodemographics"), KEYS, "inner")
    .join(spark.table(f"{SILVER}.customer_commercial_activity"), KEYS, "inner")
    .withColumn("n_products", sum(F.col(c) for c in PRODUCT_COLS))
    .withColumn("tenure_months", F.months_between("partition_month", "entry_month").cast("int"))
)

# Inner joins must not drop customer-months
check_row_count(customer_monthly, products.count(), "customer_monthly join")

In [0]:
write_gold(customer_monthly, "customer_monthly",
    "Single customer view per month. Grain: customer_id + partition_month. "
    "Joins silver customer_products, customer_sociodemographics and customer_commercial_activity. "
    "n_products = products held; tenure_months = months since first contract.")

## Monthly KPIs

In [0]:
cm = spark.table(f"{GOLD}.customer_monthly")

kpi_customers = cm.groupBy("partition_month").agg(
    F.count("*").alias("customers"),
    F.sum("is_active").alias("active_customers"),
    F.round(F.avg("is_active") * 100, 1).alias("pct_active"),
    F.round(F.avg("n_products"), 2).alias("avg_products_per_customer"),
    *[F.round(F.avg(c) * 100, 2).alias(f"pct_{c}") for c in PRODUCT_COLS],
)

kpi_sales = (
    spark.table(f"{SILVER}.sales")
    .join(spark.table(f"{SILVER}.product_description"), "product_id")
    .groupBy("sale_month", "product_family")
    .agg(
        F.count("*").alias("n_sales"),
        F.round(F.sum("net_margin"), 2).alias("net_margin_total"),
        F.round(F.avg("net_margin"), 2).alias("net_margin_avg"),
    )
)

check_unique(kpi_customers, ["partition_month"])
check_unique(kpi_sales, ["sale_month", "product_family"])

In [0]:
write_gold(kpi_customers, "kpi_customers_monthly",
    "Monthly customer base KPIs: volume, activity and product penetration (pct). Grain: partition_month.")
write_gold(kpi_sales, "kpi_sales_monthly",
    "Monthly sales KPIs by product family: number of sales and net margin. Grain: sale_month + product_family.")

## propensity_features
- **Target**: actual `pension_plan` sale in t+1 (source: `silver.sales`), not the 0→1 holdings flag.
  About half of the flag transitions are reactivations with no sale (see `00_exploration`).
- **Population at t**: never held the product, adult, not deceased, consecutive t+1 snapshot.
- Features only use information up to t.

In [0]:
product_features = [c for c in PRODUCT_COLS if c not in EXCLUDED_PRODUCTS + [TARGET]]

target_product_ids = (spark.table(f"{SILVER}.product_description")
    .filter(F.col("product_name") == TARGET).select("product_id"))
sales_next = align_sales_to_snapshot(spark.table(f"{SILVER}.sales"), target_product_ids)

eligible = (add_customer_timeline(cm, TARGET)
    .join(sales_next, KEYS, "left")
    .filter(F.col("has_next")
            & (F.col(TARGET) == 0)
            & (F.col("ever_had_before") == 0)
            & ~F.col("is_deceased")
            & (F.col("age") >= 18)))

propensity = eligible.select(
    *KEYS,
    "age", "gender", "country_id", "region_code",
    "salary", F.col("salary").isNull().cast("int").alias("salary_missing"),
    "is_active",
    F.when(F.col("has_prev"), F.col("prev_is_active")).alias("was_active_prev_month"),
    F.coalesce("segment_name", F.lit("UNKNOWN")).alias("segment"),
    F.coalesce("entry_channel", F.lit("UNKNOWN")).alias("entry_channel"),
    "tenure_months", "n_products",
    F.when(F.col("has_prev"), F.col("n_products") - F.col("prev_n_products")).alias("n_products_delta_1m"),
    *product_features,
    F.coalesce("has_sale_next_month", F.lit(0)).alias("target"),
)

In [0]:
check_unique(propensity, KEYS)
check_in_domain(propensity, "target", [0, 1])

# Leakage guards
if TARGET in propensity.columns:
    raise DataQualityError(f"Leakage: {TARGET} at t is among the features")
last_snapshot = cm.agg(F.max("partition_month")).first()[0]
if propensity.agg(F.max("partition_month")).first()[0] >= last_snapshot:
    raise DataQualityError("Last snapshot has no t+1 and must not carry a target")

# Cross-source consistency: in this population a sale and a 0->1 flag should coincide
n_mismatch = eligible.filter(F.coalesce("has_sale_next_month", F.lit(0)) != F.col("next_flag")).count()
if n_mismatch:
    print(f"WARN: {n_mismatch:,} rows where sales and holdings disagree")

In [0]:
write_gold(propensity, "propensity_features",
    "Features for pension_plan propensity. Grain: customer_id + partition_month (t). "
    "target = actual pension_plan sale in t+1 (source: silver.sales). "
    "Population: never held the product, adult, not deceased, consecutive t+1 snapshot. "
    "Flag reactivations (~50 pct of 0 to 1 transitions) excluded: they produce no sale. "
    "salary not imputed (imputation fitted on train months only) + salary_missing flag. "
    "Jan 2018 has no prior history: ever_had_before is censored for that month.")

positives = spark.table(f"{GOLD}.propensity_features").agg(F.sum("target")).first()[0]
print(f"positives (sales in t+1): {positives:,}")